In [3]:
# ----------------------------
# Section 1: Read data + merge + keep only background features + standardize missing codes
# ----------------------------

library(dplyr)
library(glmnet)
library(randomForest)
library(xgboost)

# Update the paths according to your local machine
train <- read.csv("train.csv")
background <- read.csv("background.csv")

# Remove whitespace from column names to avoid misalignment
names(train) <- trimws(names(train))
names(background) <- trimws(names(background))

# Sort background by challengeID and set rownames
background <- background[order(background$challengeID), ]
rownames(background) <- background$challengeID

# Merge datasets (inner join ensures only challengeIDs present in both are kept)
merged <- train %>% inner_join(background, by = "challengeID")

# Keep only: challengeID + gpa + all columns from background (as features)
bg_cols <- setdiff(names(background), "challengeID")
df <- merged %>% select(challengeID, gpa, all_of(bg_cols))

# Standardize missing codes (only for numeric columns)
missing_codes <- c(-9, -6, -3, -2, -1)
df <- df %>%
  mutate(across(where(is.numeric), ~ replace(., . %in% missing_codes, NA_real_)))

Warning message:
"package 'dplyr' was built under R version 4.4.3"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'glmnet' was built under R version 4.4.3"
Loading required package: Matrix

Warning message:
"package 'Matrix' was built under R version 4.4.3"
Loaded glmnet 4.1-10

Warning message:
"package 'randomForest' was built under R version 4.4.3"
randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: 'randomForest'


The following object is masked from 'package:dplyr':

    combine


Warning message:
"package 'xgboost' was built under R version 4.4.3"

Attaching package: 'xgboost'


The following object is masked from 'package:dplyr':

    slice




In [5]:
# ----------------------------
# Section 2: Data splitting + pre-processing
# ----------------------------

set.seed(123)

# Keep only rows with observed GPA
df_obs <- df %>% filter(!is.na(gpa))

# Validation set: 10%
n <- nrow(df_obs)
eval_idx <- sample.int(n, size = floor(0.10 * n))
eval_set <- df_obs[eval_idx, , drop = FALSE]
train_df <- df_obs[-eval_idx, , drop = FALSE]

# All feature columns excluding challengeID and gpa
feature_cols_all <- setdiff(names(train_df), c("challengeID", "gpa"))

# Safely convert features to numeric
to_numeric_safe <- function(x) {
  if (inherits(x, c("Date", "POSIXct", "POSIXt"))) as.numeric(x)
  else if (is.logical(x)) as.numeric(x)
  else x
}

train_df[feature_cols_all] <- lapply(train_df[feature_cols_all], to_numeric_safe)
eval_set[feature_cols_all]  <- lapply(eval_set[feature_cols_all],  to_numeric_safe)

# Keep only numeric features
feature_cols <- feature_cols_all[sapply(train_df[feature_cols_all], is.numeric)]

# Fit winsorization parameters
fit_winsor_params <- function(x) {
  x_num <- x[is.finite(x) & !is.na(x)]
  if (length(x_num) < 5) return(list(lower = -Inf, upper = Inf))
  Q1 <- quantile(x_num, 0.25)
  Q3 <- quantile(x_num, 0.75)
  IQR <- Q3 - Q1
  list(lower = Q1 - 1.5*IQR, upper = Q3 + 1.5*IQR)
}
winsor_params <- lapply(train_df[feature_cols], fit_winsor_params)

# Apply winsorization
apply_winsor <- function(x, params) pmin(pmax(x, params$lower), params$upper)
train_df[feature_cols] <- Map(apply_winsor, train_df[feature_cols], winsor_params)
eval_set[feature_cols]  <- Map(apply_winsor,  eval_set[feature_cols],  winsor_params)

# Impute missing values with training set median
medians <- vapply(train_df[feature_cols], function(z) median(z, na.rm = TRUE), numeric(1))
for (col in feature_cols) {
  train_df[[col]][is.na(train_df[[col]])] <- medians[[col]]
  eval_set[[col]][is.na(eval_set[[col]])] <- medians[[col]]
}

In [7]:
# ----------------------------
# Section 3: Feature pre-processing and feature selection
# ----------------------------

feature_cols <- setdiff(names(train_df), c("challengeID", "gpa"))
feature_cols <- feature_cols[sapply(train_df[feature_cols], is.numeric)]

# Remove all NA columns
all_na_cols <- names(train_df[feature_cols])[vapply(train_df[feature_cols], function(z) all(is.na(z)), logical(1))]
feature_cols <- setdiff(feature_cols, all_na_cols)

# Remove highly correlated columns (0.95 threshold)
cor_mat <- suppressWarnings(cor(train_df[feature_cols], use = "pairwise.complete.obs"))
high_corr_idx <- if (!any(is.na(cor_mat))) caret::findCorrelation(cor_mat, cutoff = 0.95) else integer(0)
if (length(high_corr_idx) > 0) feature_cols <- setdiff(feature_cols, colnames(cor_mat)[high_corr_idx])

# Non-finite value handling
medians <- vapply(train_df[feature_cols], function(z) median(z, na.rm = TRUE), numeric(1))
medians[is.na(medians)] <- 0
for (col in feature_cols) {
  v_tr <- train_df[[col]]; v_tr[!is.finite(v_tr)] <- NA; v_tr[is.na(v_tr)] <- medians[[col]]; train_df[[col]] <- v_tr
  v_ev <- eval_set[[col]];  v_ev[!is.finite(v_ev)]  <- NA; v_ev[is.na(v_ev)]  <- medians[[col]];  eval_set[[col]]  <- v_ev
}

#  LASSO Feature Selection
y_train <- train_df$gpa
X_train_mat <- as.matrix(train_df[feature_cols])

    set.seed(123)

cv.lasso <- glmnet::cv.glmnet(
  x = X_train_mat, y = y_train,
  alpha = 0.8, family = "gaussian",
  nfolds = 5, type.measure = "mse", standardize = TRUE
)
coef_1se <- as.matrix(coef(cv.lasso, s = "lambda.1se"))
selected_vars <- rownames(coef_1se)[coef_1se[,1] != 0]
selected_vars <- setdiff(selected_vars, "(Intercept)")
if (length(selected_vars) == 0) selected_vars <- feature_cols

stacking_train <- train_df %>% dplyr::select(dplyr::all_of(selected_vars), gpa)
stacking_eval  <- eval_set  %>% dplyr::select(dplyr::all_of(selected_vars), gpa)


In [9]:
# ----------------------------
# Section 4: Training and prediction of stacking ensembles
# ----------------------------

x_train <- stacking_train %>% dplyr::select(-gpa)
y_train <- stacking_train$gpa
x_eval  <- stacking_eval  %>% dplyr::select(-gpa)
y_eval  <- stacking_eval$gpa

# 10-fold out-of-fold (OOF) cross-validation
set.seed(123)
K <- 10
folds <- caret::createFolds(y_train, k = K, list = TRUE)

oof_rf  <- numeric(nrow(x_train))
oof_xgb <- numeric(nrow(x_train))

xgb_params <- list(
  objective = "reg:squarederror",
  eval_metric = "rmse",
  eta = 0.03,
  max_depth = 3,
  subsample = 0.7,
  colsample_bytree = 0.7,
  min_child_weight = 10,
  gamma = 1.0,
  reg_lambda = 1.5,
  reg_alpha = 0.1
)

for (k in seq_len(K)) {
  val_idx <- folds[[k]]
  tr_idx  <- setdiff(seq_len(nrow(x_train)), val_idx)
  
  # randomForest
  rf_k <- randomForest::randomForest(
    x = x_train[tr_idx, , drop = FALSE],
    y = y_train[tr_idx],
    ntree = 500,
    mtry = max(1, floor(sqrt(ncol(x_train))*0.8)),
    nodesize = 15,
    maxnodes = 50
  )
  oof_rf[val_idx] <- predict(rf_k, x_train[val_idx, , drop = FALSE])
  
  # XGBoost
  dtr <- xgb.DMatrix(as.matrix(x_train[tr_idx, , drop = FALSE]), label = y_train[tr_idx])
  dvl <- xgb.DMatrix(as.matrix(x_train[val_idx, , drop = FALSE]), label = y_train[val_idx])
  xgb_k <- xgb.train(
    params = xgb_params,
    data = dtr,
    nrounds = 3000,
    early_stopping_rounds = 100,
    watchlist = list(train = dtr, eval = dvl),
    verbose = 0
  )
  oof_xgb[val_idx] <- predict(xgb_k, xgb.DMatrix(as.matrix(x_train[val_idx, , drop = FALSE])))
}

# Meta model: Ridge Regression
meta_model <- cv.glmnet(
  x = as.matrix(data.frame(rf=oof_rf, xgb=oof_xgb)),
  y = y_train,
  alpha = 0,
  nfolds = 5
)

# Base model retraining on the entire training dataset
rf_model <- randomForest::randomForest(
  x = x_train, y = y_train,
  ntree = 700,
  mtry = max(1, floor(sqrt(ncol(x_train))*0.8)),
  nodesize = 15,
  maxnodes = 50
)

dtrain_full <- xgb.DMatrix(as.matrix(x_train), label=y_train)
deval_full  <- xgb.DMatrix(as.matrix(x_eval), label=y_eval)
xgb_model <- xgb.train(
  params = xgb_params,
  data = dtrain_full,
  nrounds = 4000,
  early_stopping_rounds = 150,
  watchlist = list(train=dtrain_full, eval=deval_full),
  verbose = 0
)

# Final prediction function
stacking_predict <- function(newdata) {
  rf_pred  <- predict(rf_model, newdata)
  xgb_pred <- predict(xgb_model, xgb.DMatrix(as.matrix(newdata)))
  meta_x   <- as.matrix(data.frame(rf=rf_pred, xgb=xgb_pred))
  p <- predict(meta_model, newx=meta_x, s="lambda.min")
  as.numeric(pmax(p, 0))  # Guarantee non-negativity
}

# Use
final_pred <- stacking_predict(x_eval)



In [11]:
# ----------------------------
# Section 5: Model Saving and Preprocessing Parameters
# ----------------------------

# Fit the Winsorised parameters and median using the training set x_train (for consistent imputation during prediction)
fit_winsor_params <- function(x) {
  x_num <- x[is.finite(x) & !is.na(x)]
  if (length(x_num) < 5) return(list(lower = -Inf, upper = Inf))
  Q1 <- stats::quantile(x_num, 0.25, na.rm = TRUE)
  Q3 <- stats::quantile(x_num, 0.75, na.rm = TRUE)
  IQR <- Q3 - Q1
  list(lower = Q1 - 1.5 * IQR, upper = Q3 + 1.5 * IQR)
}

winsor_params <- lapply(x_train, fit_winsor_params)
medians <- vapply(x_train, function(z) median(z, na.rm = TRUE), numeric(1))
medians[is.na(medians)] <- 0

# Save the model and associated parameters
model <- list(
  rf            = rf_model,
  xgb           = xgb_model,
  meta          = meta_model,
  feat          = colnames(x_train),        
  medians       = medians,                  
  winsor        = winsor_params,            
  missing_codes = c(-9, -6, -3, -2, -1),    
  background    = background                
)

# Save to file
saveRDS(model, "stacking_model.rds")



In [13]:
# ----------------------------
# Section 5: Integrated into predict.ff.gpa
# ----------------------------

predict.ff.gpa <- local({

  # Load the model only once
  model <- readRDS("stacking_model.rds")
  
  # Preprocessing function (strictly following the training phase: winsorization + median imputation)
  preprocess_rows <- function(rows) {
    X <- rows[, model$feat, drop = FALSE]
    for (i in seq_along(X)) {
      # Fill missing / non-finite values / special missing codes
      X[[i]][is.na(X[[i]]) | X[[i]] %in% model$missing_codes | !is.finite(X[[i]])] <- model$medians[i]
      # Winsorization
      bounds <- model$winsor[[i]]
      X[[i]] <- pmin(pmax(X[[i]], bounds$lower), bounds$upper)
    }
    X
  }

  # Internal cache (for single-row prediction)
  cache <- list()

  function(row) {
    id <- row$challengeID
    
    # If already predicted, return cached result
    if (!is.null(cache[[as.character(id)]])) return(cache[[as.character(id)]])
    
    # Preprocess the row
    X_mat <- as.matrix(preprocess_rows(row))
    
    # Base model predictions
    p_rf  <- predict(model$rf, X_mat)
    p_xgb <- predict(model$xgb, xgb.DMatrix(X_mat))
    
    # Stacking meta layer
    meta_x <- as.matrix(data.frame(rf = p_rf, xgb = p_xgb))
    p_final <- predict(model$meta, newx = meta_x, s = "lambda.min")
    
    # Ensure consistency with test code scale
    p_final <- as.numeric(p_final)  

    # Cache the result
    cache[[as.character(id)]] <<- p_final
    return(p_final)
  }
})

In [15]:
train = read.csv("train.csv")
background = read.csv("background.csv")
new_data = read.csv("train.csv")

# !!! Key fix: set row names → avoid freezing when new_background is completely empty
rownames(background) <- background$challengeID

# baseline: mean GPA
mean_gpa = mean(train[,'gpa'], na.rm = TRUE)

# get the GPA values for the new data
ind = !is.na(new_data[,'gpa'])
new_gpa = new_data[ind,'gpa']
new_ids = new_data[ind,'challengeID']

# !!! Key fix: index the correct rows from background
new_background = background[new_ids,]

# debug: ensure the number of rows match
print(length(new_ids))
print(nrow(new_background))

# baseline error
error_baseline = sum((new_gpa - mean_gpa)^2)

[1] "checking sizes:"
[1] 1165
[1] 1165


In [17]:
new_predictions = sapply(1:nrow(new_background), function(i) predict.ff.gpa(new_background[i, ]))

In [19]:
model_error = sum((new_predictions - new_gpa)^2)
print(1 - model_error/error_baseline)

[1] 0.3149236


First, we merged the train and background datasets using challenge_id as the key, keeping only GPA, ID, and other relevant variables.
During data cleaning, we replaced special missing codes with NA based on the codebook, split the data into training and testing sets, applied winsorization to reduce extreme values, and imputed missing data with median values.
We then used LASSO to select variables that are most predictive of GPA, and built a stacking model based on these variables. The first layer consists of two nonlinear models (Random Forest and XGBoost), and the second layer uses ridge regression as the meta-learner.
To reduce overfitting, we increased the number of K-fold OOF predictions. After the model was trained, we applied it to the test set to generate the final results.
We organized our code by dividing it into several clear sections, making the code traightforward and easier to understand.